In [ ]:
# Cell 0: Environment & seeds (debug)
import os, random
import numpy as np
import torch

# Sync CUDA errors for precise stack traces (disable after debugging)
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

# Cell 1: Imports
import pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import confusion_matrix, accuracy_score
from features import get_feature  # expects 'dwt_db4_l2'

# Cell 2: Load spectra & labels
PROC_CSV = "data/processed/ibc_processed.csv"
LABS_CSV = "data/labels_filtered.csv"

X_raw = pd.read_csv(PROC_CSV).values   # (N, 256) spectra (resampled)
y_raw = pd.read_csv(LABS_CSV)["subject_id"].values
print("Raw shapes:", X_raw.shape, y_raw.shape)

# DWT stats (db4, level=2) from the same X_raw
dwt_feat = get_feature("dwt_db4_l2")
X_dwt = dwt_feat.fit(X_raw, y_raw).transform(X_raw)   # (N, D_dwt)
print("DWT shape:", X_dwt.shape)

# Combine features: [raw || dwt]
X = np.hstack([X_raw, X_dwt]).astype(np.float32)
y = y_raw.copy()
print("Combined shape:", X.shape)

# Cell 3: One-fold stratified split (80/20)
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, val_idx = next(sss.split(X, y))
X_tr, X_va = X[train_idx], X[val_idx]
y_tr, y_va = y[train_idx], y[val_idx]

print("Train/Val:", X_tr.shape, X_va.shape)
print("Unique classes (train/val):", len(np.unique(y_tr)), len(np.unique(y_va)))

# Cell 4: Fold-local label encoding (0..C-1) and sanity checks
classes = np.unique(y_tr)
to_index = {int(c): i for i, c in enumerate(classes)}
to_label = {i: int(c) for i, c in enumerate(classes)}

def encode(labels):
    return np.vectorize(lambda t: to_index[int(t)])(labels)

def decode(indices):
    return np.vectorize(lambda i: to_label[int(i)])(indices)

y_tr_enc = encode(y_tr)
# Remove val classes missing in train (should be none in stratified split, but check)
val_mask = np.isin(y_va, classes)
if not val_mask.all():
    print("Warning: some val classes not in train; filtering those samples.")
X_va, y_va = X_va[val_mask], y_va[val_mask]
y_va_enc = encode(y_va)

print("Encoded label ranges (train/val):",
      (int(y_tr_enc.min()), int(y_tr_enc.max())),
      (int(y_va_enc.min()), int(y_va_enc.max())))
print("Class counts (train):", {int(c): int((y_tr_enc==c).sum()) for c in np.unique(y_tr_enc)})

# Cell 5: Standardization (fit on train only) on combined features
tr_mean = X_tr.mean(axis=0, keepdims=True)
tr_std = X_tr.std(axis=0, keepdims=True) + 1e-8
X_trn = (X_tr - tr_mean) / tr_std
X_val = (X_va - tr_mean) / tr_std

# Cell 6: PyTorch dataset/dataloader
from torch.utils.data import TensorDataset, DataLoader

C = len(classes)
D = X_trn.shape[1]
BATCH = 64
EPOCHS = 1000
LR = 5e-4
WD = 1e-3

Xt = torch.from_numpy(X_trn).float().to(DEVICE)
yt = torch.from_numpy(y_tr_enc).long().to(DEVICE)
Xv = torch.from_numpy(X_val).float().to(DEVICE)
yv = torch.from_numpy(y_va_enc).long().to(DEVICE)

train_dl = DataLoader(TensorDataset(Xt, yt), batch_size=BATCH, shuffle=True, drop_last=False)

# Cell 7: Define MLP (slightly larger hidden for combined input)
import torch.nn as nn
import torch.nn.functional as F

class DWTMLPNet(nn.Module):
    def __init__(self, in_dim: int, n_classes: int, hidden: int = 256, p: float = 0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(p),
            nn.Linear(hidden, hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(p),
            nn.Linear(hidden, n_classes),
        )
    def forward(self, x):
        return self.net(x)

model = DWTMLPNet(D, C, hidden=256, p=0.3).to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
crit = nn.CrossEntropyLoss()

# Cell 8: Train loop with logging & best checkpoint
best_val = float("inf")
best_state = None

def eval_model():
    model.eval()
    with torch.no_grad():
        logits = model(Xv)
        loss = crit(logits, yv).item()
        preds = logits.argmax(dim=1).cpu().numpy()
        acc = accuracy_score(y_va_enc, preds)
    return loss, acc, preds

for epoch in range(1, EPOCHS+1):
    model.train()
    running = 0.0
    for xb, yb in train_dl:
        opt.zero_grad()
        logits = model(xb)
        loss = crit(logits, yb)
        loss.backward()
        opt.step()
        running += loss.item() * xb.size(0)
    tr_loss = running / len(train_dl.dataset)
    va_loss, va_acc, _ = eval_model()
    if va_loss < best_val:
        best_val = va_loss
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    if epoch % 100 == 0 or epoch == 1:
        print(f"[{epoch:03d}] train_loss={tr_loss:.4f} val_loss={va_loss:.4f} val_acc={va_acc:.4f}")

# Load best
if best_state is not None:
    model.load_state_dict(best_state)

# Cell 9: Final evaluation and confusion matrix (decoded labels)
model.eval()
with torch.no_grad():
    val_logits = model(Xv)
    val_preds_idx = val_logits.argmax(dim=1).cpu().numpy()
val_acc = accuracy_score(y_va_enc, val_preds_idx)
val_preds_lbl = decode(val_preds_idx)

print(f"Final val accuracy: {val_acc:.4f}")
cm = confusion_matrix(y_va, val_preds_lbl, labels=classes)
print("Confusion matrix shape:", cm.shape)

# Inspect a few mistakes
mis_idx = np.where(val_preds_lbl != y_va)[0][:10]
print("First 10 misclassified indices (in val set):", mis_idx)
print("True -> Pred (decoded):", [(int(y_va[i]), int(val_preds_lbl[i])) for i in mis_idx])

# Cell 10: Sanity checks for CE target range
assert y_tr_enc.min() >= 0 and y_tr_enc.max() < C, "Train target out of range"
assert y_va_enc.min() >= 0 and y_va_enc.max() < C, "Val target out of range"
print("Label index range checks passed.")


In [ ]:
# Cell 0: Environment & seeds (debug)
import os, random
import numpy as np
import torch

# Sync CUDA errors for precise stack traces (disable after debugging)
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

# Cell 1: Imports
import pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import confusion_matrix, accuracy_score

# Cell 2: Load spectra & labels (raw only for CNN)
PROC_CSV = "data/processed/ibc_processed.csv"
LABS_CSV = "data/labels_filtered.csv"

X_raw = pd.read_csv(PROC_CSV).values.astype(np.float32)   # (N, 256) spectra (resampled)
y_raw = pd.read_csv(LABS_CSV)["subject_id"].values
print("Raw shapes:", X_raw.shape, y_raw.shape)

# Cell 3: One-fold stratified split (80/20)
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, val_idx = next(sss.split(X_raw, y_raw))
X_tr, X_va = X_raw[train_idx], X_raw[val_idx]
y_tr, y_va = y_raw[train_idx], y_raw[val_idx]

print("Train/Val:", X_tr.shape, X_va.shape)
print("Unique classes (train/val):", len(np.unique(y_tr)), len(np.unique(y_va)))

# Cell 4: Fold-local label encoding (0..C-1) and sanity checks
classes = np.unique(y_tr)
to_index = {int(c): i for i, c in enumerate(classes)}
to_label = {i: int(c) for i, c in enumerate(classes)}

def encode(labels):
    return np.vectorize(lambda t: to_index[int(t)])(labels)

def decode(indices):
    return np.vectorize(lambda i: to_label[int(i)])(indices)

y_tr_enc = encode(y_tr)
# Remove val classes missing in train (should be none in stratified split, but check)
val_mask = np.isin(y_va, classes)
if not val_mask.all():
    print("Warning: some val classes not in train; filtering those samples.")
X_va, y_va = X_va[val_mask], y_va[val_mask]
y_va_enc = encode(y_va)

print("Encoded label ranges (train/val):",
      (int(y_tr_enc.min()), int(y_tr_enc.max())),
      (int(y_va_enc.min()), int(y_va_enc.max())))
print("Class counts (train):", {int(c): int((y_tr_enc==c).sum()) for c in np.unique(y_tr_enc)})

# Cell 5: Standardization (fit on train only) on raw spectra
tr_mean = X_tr.mean(axis=0, keepdims=True)
tr_std = X_tr.std(axis=0, keepdims=True) + 1e-8
X_trn = (X_tr - tr_mean) / tr_std
X_val = (X_va - tr_mean) / tr_std

# Cell 6: PyTorch dataset/dataloader (reshape to (B, 1, 256) in forward)
from torch.utils.data import TensorDataset, DataLoader

C = len(classes)
L = X_trn.shape[1]  # 256
BATCH = 64
EPOCHS = 300
LR = 1e-3
WD = 1e-3

Xt = torch.from_numpy(X_trn).float().to(DEVICE)  # (N, 256)
yt = torch.from_numpy(y_tr_enc).long().to(DEVICE)
Xv = torch.from_numpy(X_val).float().to(DEVICE)
yv = torch.from_numpy(y_va_enc).long().to(DEVICE)

train_dl = DataLoader(TensorDataset(Xt, yt), batch_size=BATCH, shuffle=True, drop_last=False)

# Cell 7: Define SpectralCNN (1D CNN over length-256 spectra)
import torch.nn as nn
import torch.nn.functional as F

class ConvBlock1D(nn.Module):
    def __init__(self, c_in: int, c_out: int, k: int = 7, p: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(c_in, c_out, kernel_size=k, padding=k//2),
            nn.BatchNorm1d(c_out),
            nn.ReLU(inplace=True),
            nn.Dropout(p),
        )
    def forward(self, x):
        return self.net(x)

class SpectralCNN(nn.Module):
    """Input: (B, 256) float; internally unsqueezed to (B,1,256)."""
    def __init__(self, n_classes: int):
        super().__init__()
        self.feat = nn.Sequential(
            ConvBlock1D(1, 32, k=7, p=0.1),
            ConvBlock1D(32, 64, k=5, p=0.1),
            nn.MaxPool1d(2),  # 256->128
            ConvBlock1D(64, 128, k=5, p=0.1),
            nn.MaxPool1d(2),  # 128->64
            ConvBlock1D(128, 128, k=3, p=0.1),
            nn.AdaptiveAvgPool1d(1),  # -> (B,128,1)
        )
        self.cls = nn.Linear(128, n_classes)

    def forward(self, x):      # x: (B, 256)
        x = x.unsqueeze(1)     # -> (B,1,256)
        h = self.feat(x).squeeze(-1)  # (B,128)
        return self.cls(h)

model = SpectralCNN(n_classes=C).to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
crit = nn.CrossEntropyLoss()

# Cell 8: Train loop with logging & best checkpoint
best_val = float("inf")
best_state = None

def eval_model():
    model.eval()
    with torch.no_grad():
        logits = model(Xv)
        loss = crit(logits, yv).item()
        preds = logits.argmax(dim=1).cpu().numpy()
        acc = accuracy_score(y_va_enc, preds)
    return loss, acc, preds

for epoch in range(1, EPOCHS+1):
    model.train()
    running = 0.0
    for xb, yb in train_dl:
        opt.zero_grad()
        logits = model(xb)
        loss = crit(logits, yb)
        loss.backward()
        opt.step()
        running += loss.item() * xb.size(0)
    tr_loss = running / len(train_dl.dataset)
    va_loss, va_acc, _ = eval_model()
    if va_loss < best_val:
        best_val = va_loss
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    if epoch % 50 == 0 or epoch == 1:
        print(f"[{epoch:03d}] train_loss={tr_loss:.4f} val_loss={va_loss:.4f} val_acc={va_acc:.4f}")

# Load best
if best_state is not None:
    model.load_state_dict(best_state)

# Cell 9: Final evaluation and confusion matrix (decoded labels)
model.eval()
with torch.no_grad():
    val_logits = model(Xv)
    val_preds_idx = val_logits.argmax(dim=1).cpu().numpy()
val_acc = accuracy_score(y_va_enc, val_preds_idx)
val_preds_lbl = decode(val_preds_idx)

print(f"Final val accuracy: {val_acc:.4f}")
cm = confusion_matrix(y_va, val_preds_lbl, labels=classes)
print("Confusion matrix shape:", cm.shape)

# Inspect a few mistakes
mis_idx = np.where(val_preds_lbl != y_va)[0][:10]
print("First 10 misclassified indices (in val set):", mis_idx)
print("True -> Pred (decoded):", [(int(y_va[i]), int(val_preds_lbl[i])) for i in mis_idx])

# Cell 10: Sanity checks for CE target range
assert y_tr_enc.min() >= 0 and y_tr_enc.max() < C, "Train target out of range"
assert y_va_enc.min() >= 0 and y_va_enc.max() < C, "Val target out of range"
print("Label index range checks passed.")


In [ ]:
# SVD Classifier Notebook — with rank/lambda sweep

# Cell 0: Environment & seeds (debug)
import os, random
import numpy as np
import torch

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

# Cell 1: Imports
import pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import confusion_matrix, accuracy_score

# Cell 2: Load spectra & labels (raw only)
PROC_CSV = "data/processed/ibc_processed.csv"
LABS_CSV = "data/labels_filtered.csv"

X_raw = pd.read_csv(PROC_CSV).values.astype(np.float32)   # (N, 256)
y_raw = pd.read_csv(LABS_CSV)["subject_id"].values
print("Raw shapes:", X_raw.shape, y_raw.shape)

# Cell 3: One-fold stratified split (80/20)
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, val_idx = next(sss.split(X_raw, y_raw))
X_tr, X_va = X_raw[train_idx], X_raw[val_idx]
y_tr, y_va = y_raw[train_idx], y_raw[val_idx]
print("Train/Val:", X_tr.shape, X_va.shape)
print("Unique classes (train/val):", len(np.unique(y_tr)), len(np.unique(y_va)))

# Cell 4: Fold-local label encoding (0..C-1) and sanity checks
classes = np.unique(y_tr)
to_index = {int(c): i for i, c in enumerate(classes)}
to_label  = {i: int(c) for i, c in enumerate(classes)}

def encode(labels): return np.vectorize(lambda t: to_index[int(t)])(labels)
def decode(indices): return np.vectorize(lambda i: to_label[int(i)])(indices)

y_tr_enc = encode(y_tr)
val_mask = np.isin(y_va, classes)
if not val_mask.all():
    print("Warning: some val classes not in train; filtering those samples.")
X_va, y_va = X_va[val_mask], y_va[val_mask]
y_va_enc = encode(y_va)

print("Encoded label ranges (train/val):",
      (int(y_tr_enc.min()), int(y_tr_enc.max())),
      (int(y_va_enc.min()), int(y_va_enc.max())))
print("Class counts (train):", {int(c): int((y_tr_enc==c).sum()) for c in np.unique(y_tr_enc)})

# Cell 5: Standardization (fit on train only) on raw spectra
tr_mean = X_tr.mean(axis=0, keepdims=True)
tr_std  = X_tr.std(axis=0, keepdims=True) + 1e-8
X_trn = (X_tr - tr_mean) / tr_std
X_val = (X_va - tr_mean) / tr_std

# Cell 6: SVD projection helpers
from typing import Tuple, Optional, Dict

def svd_project_fixed(Xtr: np.ndarray, Xva: np.ndarray, r: int) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Project onto top-r right singular vectors of Xtr."""
    U, S, Vt = np.linalg.svd(Xtr, full_matrices=False)
    Vr = Vt[:r, :]                 # (r, D)
    Z_tr = Xtr @ Vr.T              # (N_tr, r)
    Z_va = Xva @ Vr.T              # (N_va, r)
    return Z_tr, Z_va, Vr

def fit_centroids(Z: np.ndarray, y: np.ndarray) -> Dict[int, np.ndarray]:
    cents = {}
    for c in np.unique(y):
        cents[int(c)] = Z[y == c].mean(axis=0)
    return cents

def predict_centroid(Z: np.ndarray, cents: Dict[int, np.ndarray]) -> np.ndarray:
    labels = sorted(cents.keys())
    Cmat = np.stack([cents[c] for c in labels], axis=0)  # (C, r)
    z2 = np.sum(Z**2, axis=1, keepdims=True)
    c2 = np.sum(Cmat**2, axis=1, keepdims=True).T
    d2 = z2 + c2 - 2.0 * (Z @ Cmat.T)
    idx = np.argmin(d2, axis=1)
    return np.array([labels[i] for i in idx], dtype=int)

def ridge_closed_form(Z: np.ndarray, y: np.ndarray, n_classes: int, lam: float = 1e-2) -> np.ndarray:
    """W in R^{r+1 x C} via closed-form ridge regression (one-vs-all with bias)."""
    N, r = Z.shape
    Zb = np.hstack([Z, np.ones((N, 1), dtype=Z.dtype)])  # bias
    C = n_classes
    Y = np.zeros((N, C), dtype=Z.dtype)
    Y[np.arange(N), y] = 1.0
    A = Zb.T @ Zb + lam * np.eye(r + 1, dtype=Z.dtype)
    B = Zb.T @ Y
    W = np.linalg.solve(A, B)  # (r+1, C)
    return W

def predict_ridge(Z: np.ndarray, W: np.ndarray) -> np.ndarray:
    Zb = np.hstack([Z, np.ones((Z.shape[0], 1), dtype=Z.dtype)])
    logits = Zb @ W
    return np.argmax(logits, axis=1).astype(int)

# Cell 7: Rank / Lambda sweep
RANKS   = [16, 32, 48, 64, 96, 128]
LAMBDAS = [1e-3, 3e-3, 1e-2, 3e-2, 1e-1]

best = {"acc": -1.0, "r": None, "lam": None, "clf": None}
hist = []

for r in RANKS:
    Ztr, Zva, Vr = svd_project_fixed(X_trn, X_val, r)
    # Nearest Centroid
    cents = fit_centroids(Ztr, y_tr_enc)
    yhat_nc = predict_centroid(Zva, cents)
    acc_nc = accuracy_score(y_va_enc, yhat_nc)
    hist.append({"clf": "NC", "r": r, "lam": None, "acc": float(acc_nc)})
    if acc_nc > best["acc"]:
        best = {"acc": acc_nc, "r": r, "lam": None, "clf": "NC"}

    # Ridge sweep
    for lam in LAMBDAS:
        W = ridge_closed_form(Ztr, y_tr_enc, n_classes=len(classes), lam=lam)
        yhat_rg = predict_ridge(Zva, W)
        acc_rg = accuracy_score(y_va_enc, yhat_rg)
        hist.append({"clf": "Ridge", "r": r, "lam": float(lam), "acc": float(acc_rg)})
        if acc_rg > best["acc"]:
            best = {"acc": acc_rg, "r": r, "lam": float(lam), "clf": "Ridge"}

print("Best SVD setting:", best)

# Cell 8: Final evaluation with best setting
Ztr_best, Zva_best, Vr_best = svd_project_fixed(X_trn, X_val, best["r"])
if best["clf"] == "NC":
    cents_best = fit_centroids(Ztr_best, y_tr_enc)
    yhat = predict_centroid(Zva_best, cents_best)
else:
    W_best = ridge_closed_form(Ztr_best, y_tr_enc, n_classes=len(classes), lam=best["lam"])
    yhat = predict_ridge(Zva_best, W_best)

val_acc = accuracy_score(y_va_enc, yhat)
print(f"Final val accuracy ({best['clf']} r={best['r']} lam={best['lam']}): {val_acc:.4f}")

# Cell 9: Confusion matrix & error inspection
val_preds_lbl = decode(yhat)
cm = confusion_matrix(y_va, val_preds_lbl, labels=classes)
print("Confusion matrix shape:", cm.shape)

mis_idx = np.where(val_preds_lbl != y_va)[0][:10]
print("Misclassified indices:", mis_idx)
print("True -> Pred (decoded):", [(int(y_va[i]), int(val_preds_lbl[i])) for i in mis_idx])

# Cell 10: Sanity checks for label indices
assert y_tr_enc.min() >= 0 and y_tr_enc.max() < len(classes), "Train target out of range"
assert y_va_enc.min() >= 0 and y_va_enc.max() < len(classes), "Val target out of range"
print("Label index range checks passed.")

# Optional: save sweep history
pd.DataFrame(hist).to_csv("results/svd_sweep_history.csv", index=False)
